In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("jiji_car_dataset.csv")
df['title'] = df['title'].str.replace(r"^New\s+", "", regex=True)
df.head(472)



In [ ]:
def extract_make_model_year(title):
    clean_title = re.sub(r"^New\s+", "", title, flags=re.IGNORECASE)
    pattern = r"(?P<Make>\b[a-zA-z\-]+)\s+(?P<Model>[A-Za-z0-9\-]+(?:\s[A-za-z0-9\-]+)?)?.*?(?P<Year>\d{4})(?!\d)"
    match = re.search(pattern,title)
    if match:
        return match.group("Make"), match.group("Model"), match.group("Year")
    return None, None, None

In [ ]:
df[['Make', 'Model', 'Year']] = df['title'].apply(
    lambda x: pd.Series(extract_make_model_year(x))
)



In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

In [ ]:
df[df['make'].isna() & df["model"].isna()]

In [ ]:
#dropping nan data 
df= df.dropna()
df.shape

In [ ]:
#clean the price column

df['price'] = (
    df['price'].astype(str).str.replace("₦", "", regex=False).str.replace(",", "", regex=False).str.strip()
)

df['price'] = pd.to_numeric(df['price'], errors='coerce')



In [ ]:
df['price'].describe()

In [ ]:
df['price'].head(10)

In [ ]:
df = df[df['price'] > 0]

In [ ]:
#Cars with the highest price
df.nlargest(10, "price")[['title','Make', 'Model', 'Year', 'price']]

In [ ]:
# standardized the categorical column
categorical_cols = ["Make", "Model", "condition", "transmission"]

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()


In [ ]:
df.head()

In [ ]:
# dropping the old make, model and year
df = df.drop(columns=['make','model','year'])

In [ ]:
df.head()

In [ ]:
df = df.rename(columns={
    'title': "Title",
    'condition': 'Condition',
    'transmission': 'Transmission',
    'price': 'Price'
})

df.head()

In [ ]:
df["Condition"] = df['Condition'].str.title()
df['Transmission'] = df['Transmission'].str.title()

df.head()

In [ ]:
df.to_csv("New_clean_jiji_automobile.csv", index=False)

In [ ]:
df['Make'].value_counts()

In [ ]:
df['Year'].describe()

In [ ]:
df['Year'] = pd.to_numeric(df['Year'])
df = df[(df['Year'] >= 1980) & (df['Year'] < 2025)]

In [ ]:
df['Year']
df.to_csv("New_clean_jiji_automobile.csv", index=False)

## NOW EDA

In [ ]:
# Most common car brand
make_counts = df['Make'].value_counts()
make_counts.head(10)

In [ ]:
# Bar chart
plt.figure(figsize=(12,6))

make_counts.head(15).plot(kind="bar")

plt.title("Most Common Car Brands")
plt.xlabel("Car Make")
plt.ylabel("Number of Listing")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Most common model per brand
df["Model"].value_counts().head(15)

In [ ]:
popular_models = (
    df.groupby(["Make", "Model"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

popular_models.head(20)

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(df["Year"], bins=20)

plt.title("Distribution of Car Manufacturing Years")
plt.xlabel("Year")
plt.ylabel("Number of Cars")

plt.tight_layout()
plt.show()

In [ ]:
# Average price per brands
avg_price_make = (
    df.groupby("Make")["Price"]
    .mean()
    .sort_values(ascending=False)
)

avg_price_make.head(15)

In [ ]:
plt.figure(figsize=(12, 6))

avg_price_make.head(15).plot(kind="bar")

plt.title("Average Car Price by Brand")
plt.xlabel("Make")
plt.ylabel("Average Price (₦)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# median make
avg_price_model = (
    df.groupby(["Make", "Model"])["Price"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)

avg_price_model.head(20)


In [ ]:
# condition vs price
df.groupby("Condition")["Price"].agg(
    ["count", "mean", "median"]
)

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x="Condition",
    y="Price"
)

plt.title("Car Price Distribution by Condition")
plt.xlabel("Condition")
plt.ylabel("Price (₦)")

plt.tight_layout()
plt.show()

In [ ]:
# Transmission vs price
df.groupby("Transmission")["Price"].agg(
    ["count", "mean", "median"]
)

In [ ]:
# chart to visualize transmission and price 
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="Transmission",
    y="Price"
)

plt.title("Car Prices by Transmission Type")
plt.xlabel("Transmission")
plt.ylabel("Price (₦)")

plt.tight_layout()
plt.show()

In [ ]:
#Year vs price 
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Year",
    y="Price",
    hue="Condition"
)

plt.title("Car Year vs Price")
plt.xlabel("Year")
plt.ylabel("Price (₦)")

plt.tight_layout()
plt.show()

In [ ]:
correlation = df["Year"].corr(df["Price"])

print("Correlation between year and price:", correlation)

In [ ]:
# correlation heat map
numeric_cols = ["Year", "Price"]

correlation_matrix = df[numeric_cols].corr()

correlation_matrix

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.tight_layout()
plt.show()

## Summary of Findings

- **Which brands cost the most?** Rolls-Royce and Land Rover show the highest average prices, but very few of them were listed (under 30 cars each), so we can't trust that number — too small a sample to mean anything solid. Among brands with enough listings to trust the numbers, Mercedes-Benz (₦69.9M average, 339 cars) and Lexus (₦49.5M average, 277 cars) are the real price leaders. Toyota's average price is lower (₦29M), but it has way more listings than any other brand (785 cars) — so Toyota isn't the most expensive, it's just the most common/most sold.

- **Foreign Used vs Local Used — which costs more?** Foreign Used cars average ₦43.8M, Local Used average ₦17.3M. That's about 2.5x more expensive for Foreign Used. Brand New cars average ₦273M but there were only 90 of them listed, so that number can swing easily and isn't very reliable — small sample.

- **Does the car's year affect price?** Yes, but not perfectly. We measured this with a correlation score of 0.44 (correlation just means: how strongly two things move together, on a scale of 0 to 1 — 0.44 is a "moderate" relationship, not super strong). When we grouped cars into 5-year chunks though, the pattern is very clear: cars from 2000–2004 average ₦5M, 2010–2014 average ₦17M, 2015–2019 average ₦34M, and 2020–2024 average ₦121M. So older cars really do lose value, step by step.

- **Automatic vs Manual.** Almost all the cars for sale are automatic — 1,924 out of 1,988 total (that's 97%). Automatic cars also cost more on average (₦44M) compared to manual (₦16.1M). Manual cars are rare and mostly cheaper, older cars.

- **Which model holds its value best — Hilux, RX 350, or Accord?** Toyota Hilux wins — average price ₦72.2M, and the cars are fairly recent (average year 2019.6). Lexus RX 350 comes next — ₦39.9M average, slightly older cars on average (2014.9). Honda Accord is lowest — ₦8.7M average, and also the oldest of the three (2010.5). So Accord loses value the most over time; Hilux keeps its value the best.

## Business Insights & Recommendations

- **Best-selling brand/model overall:** Toyota is the most popular brand by far (most cars listed = most in-demand). If you're picking one specific model that performs best, it's the Toyota Hilux — it sells for a high price even as a used car.

- **Which category brings in the most money:** Foreign Used cars. They have more listings than Local Used (1,066 vs 832) AND cost more per car — so together, they bring in more total money to the market than Local Used cars do.

- **When were most cars made:** Most of the cars listed were made between 2015–2019 (558 cars), closely followed by 2010–2014 (555 cars). Newer cars (2020 and later) are fewer in number, but they sell for the highest prices.

- **What to do about the weaker segments (Local Used, Manual cars):**
  1. Local Used cars sell in high numbers but at low prices — instead of competing on cheap price, add things like a warranty or a proper inspection to make them worth more.
  2. Manual cars are now a tiny part of the market (only 3%) — it may not be worth spending much effort sourcing more of them.
  3. Focus on getting more automatic, Foreign Used cars from 2015 onward — this is the combination that's already proven to sell for the most.